# 2.0 Выбор критерия по итогам СТАВОК (не по трейну)

Вход: файл ставок из ноутбука 1 (вкладка `curr_games`; 1, 2 или 3 фильтра - задаёт пользователь).

Один фильтр по ставкам на весь профиль (CLAY либо HARD_GRASS), без деления на группы.

Для каждого года Y накопленные ставки FIRST_YEAR..Y-1 сканируются по колонкам F-P (обе стороны `<`/`>`) и по `k2` (только строго `k2 <`). Результат - ранжированный список (формат вкладок `CLAY2021-2021`, `CLAY2021-2022`, ...).

Фильтр, выбранный по ставкам прошлых лет, накладывается на ставки года Y (вкладка `curr_games_4filter`). Правило выбора задаёт `RULE`: `"k2"` - лучший строгий `k2 <` (по умолчанию), `"rank"` - строка `RANK` общего ранжированного списка.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
import yaml

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CFG = yaml.safe_load(open(ROOT / "data" / "config.yaml", encoding="utf-8"))
F_TO_P = CFG["threshold_search"]["criteria_columns_F_to_P"]
MIN_K2 = CFG["threshold_search"]["min_matches_under_threshold"]
SPLIT = CFG["threshold_search"]["matches_split"]

PROFILE = "clay"                     # "clay" или "hard_grass"
BETS_FILE = ROOT / "data" / "processed" / "CLAY_3filters.xlsx"
BETS_SHEET = "curr_games"
FIRST_YEAR = 2021
MIN_BETS = 100                       # минимум ставок под порогом
RULE = "k2"                          # "k2" или "rank"
RANK = 0                             # используется при RULE = "rank" (0 = лучший)
TOP_N = 3                            # сколько верхних фильтров показать в проверке
OUT = ROOT / "data" / "processed" / f"BEST_BET_CRIT_{PROFILE}.xlsx"
OUT_BETS = ROOT / "data" / "processed" / (BETS_FILE.stem + "_with_4filter.xlsx")

In [ ]:
bets = pd.read_excel(BETS_FILE, sheet_name=BETS_SHEET)
bets["win2"] = np.where(bets["win"] == 0, 100.0, -100.0 / (bets["k2"] - 1))
COLS = [c for c in F_TO_P if c in bets.columns] + ["k2"]
years = sorted(bets.eval_year.unique()); years

In [ ]:
def scan(d, min_n=MIN_BETS):
    """Лучший порог по каждой колонке и стороне. k2 только '<' (строго)."""
    base = d["win2"].sum(); out = []
    for col in COLS:
        x = d[col].values; ok = ~np.isnan(x)
        xs, vs = x[ok], d["win2"].values[ok]
        o = np.argsort(xs, kind="stable"); xs, vs = xs[o], vs[o]
        cs = np.concatenate([[0], np.cumsum(vs)]); N = len(xs)
        u, first = np.unique(xs, return_index=True); last = np.append(first[1:], N)
        for op in (["<"] if col == "k2" else ["<", ">"]):
            n, s = (first, cs[first]) if op == "<" else (N - last, cs[N] - cs[last])
            m = n >= min_n
            if m.any():
                i = np.argmax(np.where(m, s, -1e18))
                out.append(dict(column=col, op=op, threshold=u[i], matches=int(n[i]), sum_win2=s[i],
                                roi=s[i] / n[i], pool_matches=N, pool_sum=base))
    return pd.DataFrame(out).sort_values("sum_win2", ascending=False).reset_index(drop=True)

def apply_crit(d, row):
    return d[d[row["column"]] < row["threshold"]] if row["op"] == "<" else d[d[row["column"]] > row["threshold"]]

def pick(tab):
    """Выбор фильтра из ранжированного списка по правилу RULE."""
    if RULE == "k2":
        return tab[tab.column == "k2"].iloc[0]
    return tab.iloc[RANK]

TABLES = {f"{PROFILE.upper()}{FIRST_YEAR}-{Y}": scan(bets[bets.eval_year <= Y]) for Y in years[:-1]}
list(TABLES)

In [ ]:
# Ранжированный список по накопленным ставкам прошлых лет -> фильтр на ставки следующего сезона
four, chk, chosen = [], [], []
for Y in years[:-1]:
    tab = TABLES[f"{PROFILE.upper()}{FIRST_YEAR}-{Y}"]; nxt = bets[bets.eval_year == Y + 1]
    row = pick(tab)
    z = apply_crit(nxt, row)
    four.append(z.assign(bet_filter=f"{row.column} {row.op} {row.threshold:.4g}", bet_filter_from=f"{FIRST_YEAR}-{Y}"))
    chosen.append(dict(applied_to_year=Y + 1, from_bets=f"{FIRST_YEAR}-{Y}", column=row.column, op=row.op,
                       threshold=row.threshold, train_sum=row.sum_win2, bets_before=len(nxt), sum_before=nxt.win2.sum(),
                       bets_after=len(z), sum_after=z.win2.sum()))
    for rank, (_, r_) in enumerate(tab.head(TOP_N).iterrows(), 1):
        zz = apply_crit(nxt, r_)
        chk.append(dict(train_period=f"{FIRST_YEAR}-{Y}", rank=rank, column=r_.column, op=r_.op, threshold=r_.threshold,
                        train_sum=r_.sum_win2, test_year=Y + 1, test_base=nxt.win2.sum(), test_filtered=zz.win2.sum(), test_bets=len(zz)))
BETS4 = pd.concat(four); APPLIED = pd.DataFrame(chosen); CHECK = pd.DataFrame(chk)
print(APPLIED[["bets_before", "sum_before", "bets_after", "sum_after"]].sum().round(1).to_dict())
APPLIED.round(2)

In [ ]:
# Сравнение: ставки без и с фильтром по ставкам (2022+; первый год не фильтруется - нет истории)
base = bets[bets.eval_year > FIRST_YEAR]
pd.DataFrame({"bets": [len(base), len(BETS4)], "sum_win2": [base.win2.sum(), BETS4.win2.sum()],
              "roi": [base.win2.mean(), BETS4.win2.mean()]}, index=["без фильтра по ставкам", "с фильтром по ставкам"]).round(2)

In [ ]:
with pd.ExcelWriter(OUT) as xw:
    for name, t in TABLES.items(): t.to_excel(xw, sheet_name=name[:31], index=False)
    APPLIED.to_excel(xw, sheet_name="applied_bet_filter", index=False)
    CHECK.to_excel(xw, sheet_name="walk_forward_check", index=False)
with pd.ExcelWriter(OUT_BETS) as xw:
    bets.to_excel(xw, sheet_name="curr_games", index=False)
    BETS4.drop(columns=[], errors="ignore").to_excel(xw, sheet_name="curr_games_4filter", index=False)
OUT, OUT_BETS